# Between-group differences in improvement — statistical validation

For every figure in the article the question is the same: **is the improvement (or outcome)
*different between the groups the figure shows*, how big is that difference, and which
contrast is the largest?**  Whether an improvement exists pre→post is assumed — we test only
whether groups *differ* in it.

Each figure → one between-group test on the relevant metric:

| Figure family | Metric compared between groups | Test |
|---|---|---|
| `pre_post_*` | **raw gain** (post − pre) | 2 groups: Mann–Whitney U · k groups: Kruskal–Wallis |
| `pre_post_trad_scale` | **grade-level change** (ordinal) | Kruskal–Wallis |
| `hake_gain_*` | **Hake normalized gain** | Mann–Whitney U / Kruskal–Wallis |
| `mathew_*` | raw gain, **High vs Low** starters | Mann–Whitney U (gap pre→post) |
| `mejora_*` | **improvement category mix** | χ² of independence |
| `self_confidence_comparison` | **self-efficacy gain** (ta post − pre) | Mann–Whitney U |
| `tcc_*` | **post cognitive load** (no pre exists) | Mann–Whitney U / Kruskal–Wallis |

**Every result reports:** the p-value, an **effect size** with magnitude, **which group
improved most**. Effect sizes are stored on a common 0–1 `strength`
scale (|rank-biserial *r*| of the biggest contrast, or Cramér's V).

---
## Setup — identical data loading & filtering to `paper_results.ipynb`

In [1]:
import os, sys
import polars as pl
from __future__ import annotations
import itertools
import numpy as np
import polars as pl
from scipy import stats
from statsmodels.stats.multitest import multipletests
from statsmodels.stats.contingency_tables import SquareTable


ALPHA = 0.05
RESULTS: list[dict] = []
V3_ORDER = ["Low", "Medium", "High"]           # Low = worst, High = best baseline
V3_RANK = {c: i for i, c in enumerate(V3_ORDER)}

script_path = os.getcwd()
project_path = os.path.join(script_path, '..', '..', '..')
data_path = os.path.join(project_path, 'data', 'combined',
                         'processed_forms_interactions_data.parquet')

sys.path.append(project_path)
from src.utils.educational_impact.educational_impact_analysis import (
    rename_df
)

from src.utils.paper.statistical_validation import (
    ALPHA, TRAD, GRADE_RANK,
    _with_metric, _num, _sig, _mag_r, _mag_eps, _rb, _reg,
    _cramers_v, _matched_rc, _mag, _contingency, _print_pct,
    _target_vs_others, _print_contrasts,
    reset_results, diff_2, diff_k, diff_cat, matthew, summary,
    paired_distribution, test_expa_trad_scale, test_expaa_pre_v3,
    test_expa_ta_post, test_mejora_across_baseline_strata,
)

forms_interactions_df = rename_df(pl.read_parquet(data_path))

problematic_ids = forms_interactions_df.filter(
    pl.col('grupo') == 'experimental', pl.col('chat_freq_use').is_null())['id'].to_list()
q25 = forms_interactions_df['score_tc_units_hake_gain'].quantile(0.25)
q75 = forms_interactions_df['score_tc_units_hake_gain'].quantile(0.75)
lb = q25 - 1.5 * (q75 - q25)
problematic_ids += forms_interactions_df.filter(pl.col('score_tc_units_hake_gain') < lb)['id'].to_list()
problematic_ids += forms_interactions_df.filter(pl.col('score_tc_pre') == 1)['id'].to_list()
problematic_ids = list(set(problematic_ids))

forms_interactions_df = forms_interactions_df.filter(~pl.col('id').is_in(problematic_ids))
no_efecto_techo_df = forms_interactions_df.filter(pl.col('score_tc_pre') < 0.8)


---
## 1 · Baseline balance (`tc_pre.pdf`, `ta_pre.pdf`)

Not an improvement — a check that the arms start equal. **Want NO difference.**

### Figure 1 and Figure 2

In [2]:
diff_2("tc_pre.pdf", "grupo", "baseline knowledge", forms_interactions_df,
       "grupo", "experimental", "control", metric="score_tc_pre", baseline=True)

diff_2("ta_pre.pdf", "grupo", "baseline self-efficacy", forms_interactions_df,
       "grupo", "experimental", "control", metric="score_ta_pre", baseline=True)

█ tc_pre.pdf   ·  grouping: grupo   ·  metric: baseline knowledge
   Q: are experimental and control balanced at baseline? (want NO difference)
   Mann-Whitney U   n=183   median experimental=+0.600 vs control=+0.600  (Δ=+0.000)
   U = 4330.0   p = 0.6704 n.s.   rank-biserial r = +0.036 (negligible)
   → baseline balanced (no significant difference).

█ ta_pre.pdf   ·  grouping: grupo   ·  metric: baseline self-efficacy
   Q: are experimental and control balanced at baseline? (want NO difference)
   Mann-Whitney U   n=183   median experimental=+0.500 vs control=+0.430  (Δ=+0.070)
   U = 5461.5   p = 0.0003 ***   rank-biserial r = +0.307 (medium)
   → ⚠ groups DIFFER at baseline — potential confound.



---
## 2 · Knowledge improvement (raw gain = post − pre) — does it differ between groups?

The `pre_post_*` figures: we compare the **raw gain** across the groups each one shows.

### Figure 3 (a)

**`pre_post_comparison.pdf`** — experimental vs control.

In [3]:
diff_2("pre_post_comparison.pdf", "grupo", "raw gain (post−pre)", forms_interactions_df,
       "grupo", "experimental", "control", pre="score_tc_pre", post="score_tc_post")

█ pre_post_comparison.pdf   ·  grouping: grupo   ·  metric: raw gain (post−pre)
   Q: is the improvement different between experimental and control?
   Mann-Whitney U   n=183   median experimental=+0.100 vs control=+0.200  (Δ=-0.100)
   U = 3691.0   p = 0.1690 n.s.   rank-biserial r = -0.117 (small)
   → no significant between-group difference (small effect).



### Figure 3 (b)

**`pre_post_quality.pdf`** — across interaction quality (`grupo_segmented_v4`).

In [4]:
diff_k("pre_post_quality.pdf", "v4", "raw gain (post−pre)", forms_interactions_df,
       "grupo_segmented_v4", pre="score_tc_pre", post="score_tc_post")

█ pre_post_quality.pdf   ·  grouping: grupo_segmented_v4   ·  metric: raw gain (post−pre)
   Q: does raw gain (post−pre) differ across the 4 groups?
   Kruskal-Wallis   H = 2.886, df = 3, n = 183   p = 0.4095 n.s.   ε² = -0.001 (negligible)
   group medians: ExpNotUsed=+0.100, ExpA=+0.100, control=+0.200, ExpB=+0.200
   highest: control (+0.200)   |   lowest: ExpNotUsed (+0.100)
   BIGGEST pairwise difference: ExpNotUsed vs control   U = 1027.0, r = -0.199 (small), p-Holm = 0.6738 n.s.
   (no pair survives Holm correction)



### Figure 3 (c)

**`pre_post_freq_quality.pdf`** — across freq×quality (`grupo_segmented_v3`).

In [5]:
diff_k("pre_post_freq_quality.pdf", "v3", "raw gain (post−pre)", forms_interactions_df,
       "grupo_segmented_v3", pre="score_tc_pre", post="score_tc_post")

█ pre_post_freq_quality.pdf   ·  grouping: grupo_segmented_v3   ·  metric: raw gain (post−pre)
   Q: does raw gain (post−pre) differ across the 6 groups?
   Kruskal-Wallis   H = 4.739, df = 5, n = 183   p = 0.4485 n.s.   ε² = -0.001 (negligible)
   group medians: ExpNotUsed=+0.100, ExpAA=+0.100, ExpBA=+0.200, ExpBB=+0.200, control=+0.200, ExpAB=+0.200
   highest: ExpBA (+0.200)   |   lowest: ExpNotUsed (+0.100)
   BIGGEST pairwise difference: ExpAA vs ExpBA   U = 49.5, r = -0.365 (medium), p-Holm = 1.0000 n.s.
   (no pair survives Holm correction)



**`score_tc_pre_V3`** - does ExpAA have a HIGHER baseline than the other v4 groups?

In [6]:
test_expaa_pre_v3(forms_interactions_df, target="ExpAA")

TEST 1 - Does 'ExpAA' have a higher baseline (score_tc_pre) than the other grupo_segmented_v3 groups?
   Metric : Continuous baseline score (score_tc_pre)
   Test   : Mann-Whitney U comparing the target group against each
            remaining group (Holm-adjusted p-values).
            Positive rank-biserial correlation (r) indicates that
            the target group has a higher baseline.

     contrast                          n_t  n_c   med(t)  med(c)        U        r         p    p-Holm   effect
     ExpAA vs ExpAB                  12   24     0.70    0.60    200.0   +0.389    0.0580    0.2460 n.s.  medium higher
     ExpAA vs ExpBA                  12   13     0.70    0.60    113.5   +0.455    0.0492    0.2460 n.s.  medium higher
     ExpAA vs ExpBB                  12   12     0.70    0.60     98.5   +0.368    0.1255    0.2460 n.s.  medium higher
     ExpAA vs ExpNotUsed             12   27     0.70    0.60    214.5   +0.324    0.1044    0.2460 n.s.  medium higher
     ExpAA vs

{'target': 'ExpAA',
 'contrasts': [{'comparator': 'ExpAB',
   'n_t': 12,
   'n_c': 24,
   'med_t': 0.7000000000000001,
   'med_c': 0.6000000000000001,
   'U': 200.0,
   'r': np.float64(0.38888888888888884),
   'p': np.float64(0.05795143113128407),
   'p_holm': np.float64(0.2460378223693989)},
  {'comparator': 'ExpBA',
   'n_t': 12,
   'n_c': 13,
   'med_t': 0.7000000000000001,
   'med_c': 0.6000000000000001,
   'U': 113.5,
   'r': np.float64(0.45512820512820507),
   'p': np.float64(0.04920756447387978),
   'p_holm': np.float64(0.2460378223693989)},
  {'comparator': 'ExpBB',
   'n_t': 12,
   'n_c': 12,
   'med_t': 0.7000000000000001,
   'med_c': 0.6000000000000001,
   'U': 98.5,
   'r': np.float64(0.3680555555555556),
   'p': np.float64(0.1255064714374691),
   'p_holm': np.float64(0.2460378223693989)},
  {'comparator': 'ExpNotUsed',
   'n_t': 12,
   'n_c': 27,
   'med_t': 0.7000000000000001,
   'med_c': 0.6000000000000001,
   'U': 214.5,
   'r': np.float64(0.3240740740740742),
   'p': n

### Figure 4

In [7]:
# ==============================================================================
# pre_post_trad_scale.pdf — does ExpA improve more than the other v4 categories?
#
#   Step 1 (pre)  : chi-square of grade-band × grupo_segmented_v4
#                   → want NON-significant  (homogeneous starting point)
#   Step 2a (post): chi-square of grade-band × grupo_segmented_v4
#                   → want significant     (groups now differ)
#   Step 2b (post): Mann-Whitney U on the ordinal grade rank
#                   (Excellent=4 … Fail=0), ExpA vs each other v4 level,
#                   Holm-adjusted   → positive r ⇒ ExpA holds better grades.
# ==============================================================================

ALPHA = 0.05
TRAD = ['Excellent', 'Very Good', 'Good', 'Pass', 'Fail']            # best → worst
GRADE_RANK = {c: (len(TRAD) - 1 - i) for i, c in enumerate(TRAD)}    # Excellent=4 … Fail=0

test_expa_trad_scale(forms_interactions_df, target="ExpA")

pre_post_trad_scale.pdf — does 'ExpA' improve more than the other grupo_segmented_v4 categories?

(1) PRE — is the grade distribution homogeneous across the 4 groups?
      test : Chi-square of independence   (want NON-significant p)
   PRE distribution (row % within group)
                    Excellent   Very Good        Good        Pass        Fail
               ExpA        4.0%       36.0%       36.0%       12.0%       12.0%   n=25
               ExpB        5.6%       30.6%       27.8%       16.7%       19.4%   n=36
         ExpNotUsed        0.0%       33.3%       33.3%       18.5%       14.8%   n=27
            control        6.3%       30.5%       26.3%       16.8%       20.0%   n=95
     χ² = 4.141, df = 12, n = 183   p = 0.9808 n.s.   Cramer's V = 0.087 (negligible)
     note: 40% of cells expected <5 (χ² approximate)
     → baseline HOMOGENEOUS ✓

(2a) POST — do the 4 groups differ overall at post?
      test : Chi-square of independence   (want significant p)
   POST distri

---
## 3 · Matthew effect — is the High–Low gap bigger at post? (`mathew_control`, `mathew_experimental`)

Within each arm we compare the **gain of High vs Low starters**. The gap widens only if High
gain > Low gain; the output prints the median gap at pre and at post.

### Figure 5 (a) and Figure 5 (b)

In [8]:
matthew("mathew_control.pdf",
        forms_interactions_df.filter(pl.col("grupo") == "control"),
        "score_tc_pre", "score_tc_post")

matthew("mathew_experimental.pdf",
        forms_interactions_df.filter(pl.col("grupo") == "experimental"),
        "score_tc_pre", "score_tc_post")

█ mathew_control.pdf   ·  grouping: High vs Low baseline   ·  metric: raw gain
   Q: is the improvement different for High vs Low starters?
   Mann-Whitney U   n=70   gain High=+0.100 vs Low=+0.400
   gap High−Low: pre=+0.400 → post=+0.100
   U = 174.5   p = 0.0000 ***   rank-biserial r = -0.715 (large)
   → Low starters gain MORE → gap narrows (compensatory).

█ mathew_experimental.pdf   ·  grouping: High vs Low baseline   ·  metric: raw gain
   Q: is the improvement different for High vs Low starters?
   Mann-Whitney U   n=60   gain High=+0.050 vs Low=+0.300
   gap High−Low: pre=+0.350 → post=+0.100
   U = 53.0   p = 0.0000 ***   rank-biserial r = -0.882 (large)
   → Low starters gain MORE → gap narrows (compensatory).



---
## 4 · Normalized (Hake) gain — does it differ between groups?

The Hake gain corrects for starting level; this is the cleanest improvement metric.

### Figure 6 (a)

**`hake_gain_comparison.pdf`** — experimental vs control  *(headline)*.

In [9]:
diff_2("hake_gain_comparison.pdf", "grupo", "Hake gain", forms_interactions_df,
       "grupo", "experimental", "control", metric="score_tc_hake_gain")

█ hake_gain_comparison.pdf   ·  grouping: grupo   ·  metric: Hake gain
   Q: is the improvement different between experimental and control?
   Mann-Whitney U   n=183   median experimental=+0.400 vs control=+0.500  (Δ=-0.100)
   U = 3437.0   p = 0.0374 *   rank-biserial r = -0.178 (small)
   → control is higher on Hake gain — difference is small.



### Figure 6 (b)

**`hake_gain_quality.pdf`** — across interaction quality (`grupo_segmented_v4`).

In [10]:
diff_k("hake_gain_quality.pdf", "v4", "Hake gain", forms_interactions_df,
       "grupo_segmented_v4", metric="score_tc_hake_gain")

█ hake_gain_quality.pdf   ·  grouping: grupo_segmented_v4   ·  metric: Hake gain
   Q: does Hake gain differ across the 4 groups?
   Kruskal-Wallis   H = 5.918, df = 3, n = 183   p = 0.1157 n.s.   ε² = 0.016 (small)
   group medians: ExpA=+0.500, ExpB=+0.367, control=+0.500, ExpNotUsed=+0.333
   highest: ExpA (+0.500)   |   lowest: ExpNotUsed (+0.333)
   BIGGEST pairwise difference: control vs ExpNotUsed   U = 1607.0, r = +0.253 (small), p-Holm = 0.2693 n.s.
   (no pair survives Holm correction)



### Figure 7 (a)

**`hake_gain_frq_quality.pdf`** — across freq×quality (`grupo_segmented_v3`).

In [11]:
diff_k("hake_gain_frq_quality.pdf", "v3", "Hake gain", forms_interactions_df,
       "grupo_segmented_v3", metric="score_tc_hake_gain")

█ hake_gain_frq_quality.pdf   ·  grouping: grupo_segmented_v3   ·  metric: Hake gain
   Q: does Hake gain differ across the 6 groups?
   Kruskal-Wallis   H = 7.200, df = 5, n = 183   p = 0.2062 n.s.   ε² = 0.012 (small)
   group medians: ExpBA=+0.500, ExpAB=+0.367, ExpAA=+0.416, control=+0.500, ExpNotUsed=+0.333, ExpBB=+0.343
   highest: ExpBA (+0.500)   |   lowest: ExpNotUsed (+0.333)
   BIGGEST pairwise difference: ExpBA vs ExpNotUsed   U = 239.5, r = +0.365 (medium), p-Holm = 0.9135 n.s.
   (no pair survives Holm correction)



### Figure 7 (b)

**`hake_gain_frq_quality_no_ceiling_effect.pdf`** — same, `pre < 0.8` only.

In [12]:
diff_k("hake_gain_frq_quality_no_ceiling_effect.pdf", "v3", "Hake gain", no_efecto_techo_df,
       "grupo_segmented_v3", metric="score_tc_hake_gain")

█ hake_gain_frq_quality_no_ceiling_effect.pdf   ·  grouping: grupo_segmented_v3   ·  metric: Hake gain
   Q: does Hake gain differ across the 6 groups?
   Kruskal-Wallis   H = 5.448, df = 5, n = 145   p = 0.3636 n.s.   ε² = 0.003 (negligible)
   group medians: ExpNotUsed=+0.367, ExpAB=+0.400, control=+0.600, ExpBA=+0.500, ExpAA=+0.500, ExpBB=+0.343
   highest: control (+0.600)   |   lowest: ExpBB (+0.343)
   BIGGEST pairwise difference: ExpBA vs ExpBB   U = 71.5, r = +0.300 (medium), p-Holm = 1.0000 n.s.
   (no pair survives Holm correction)



---
## 5 · Improvement-category mix — do proportions differ between groups?

`mejora_hake_gain_v2` = Improve / Not Improve / Worsen. χ² of independence + Cramér's V;
the output also prints the **% Improve per group** and the best/worst group.

### Figure 12 (a)

**`mejora_comparison.pdf`** — by arm (`grupo`).

In [13]:
diff_cat("mejora_comparison.pdf", "grupo", forms_interactions_df,
         "mejora_hake_gain_v2", "grupo", order=['Improve', 'Not Improve', 'Worsen'])

█ mejora_comparison.pdf   ·  grouping: grupo   ·  metric: mejora_hake_gain_v2 (proportions)
   Q: does the improvement-category mix differ between groups?
   Chi-square   χ² = 1.849, df = 2, n = 183   p = 0.3968 n.s.   Cramér's V = 0.101 (small)
   %Improve by group: control=78%, experimental=72%
   most improving: control (78%)   |   least: experimental (72%)
   → No significant difference in the improvement mix.



### Figure 12 (b)

**`mejora_quality.pdf`** — by interaction quality (`grupo_segmented_v4`).

In [14]:
diff_cat("mejora_quality.pdf", "v4", forms_interactions_df,
         "mejora_hake_gain_v2", "grupo_segmented_v4", order=['Improve', 'Not Improve', 'Worsen'])

█ mejora_quality.pdf   ·  grouping: grupo_segmented_v4   ·  metric: mejora_hake_gain_v2 (proportions)
   Q: does the improvement-category mix differ between groups?
   Chi-square   χ² = 4.664, df = 6, n = 183   p = 0.5876 n.s.   Cramér's V = 0.113 (small)
   %Improve by group: ExpA=80%, ExpB=69%, ExpNotUsed=67%, control=78%
   most improving: ExpA (80%)   |   least: ExpNotUsed (67%)
   note: 50% of cells expected <5 (χ² approximate)
   → No significant difference in the improvement mix.



### Figure 13 (a)

**`mejora_frq_quality.pdf`** — by freq×quality (`grupo_segmented_v3`).

In [15]:
diff_cat("mejora_frq_quality.pdf", "v3", forms_interactions_df,
         "mejora_hake_gain_v2", "grupo_segmented_v3", order=['Improve', 'Not Improve', 'Worsen'])

█ mejora_frq_quality.pdf   ·  grouping: grupo_segmented_v3   ·  metric: mejora_hake_gain_v2 (proportions)
   Q: does the improvement-category mix differ between groups?
   Chi-square   χ² = 12.829, df = 10, n = 183   p = 0.2334 n.s.   Cramér's V = 0.187 (small)
   %Improve by group: ExpAA=67%, ExpAB=67%, ExpBA=92%, ExpBB=75%, ExpNotUsed=67%, control=78%
   most improving: ExpBA (92%)   |   least: ExpAA (67%)
   note: 56% of cells expected <5 (χ² approximate)
   → No significant difference in the improvement mix.



### Figure 13 (b)

**`mejora_frq_quality_ceiling_effect.pdf`** — by freq×quality, `pre < 0.8` subset.

In [16]:
diff_cat("mejora_frq_quality_ceiling_effect.pdf", "v3", no_efecto_techo_df,
         "mejora_hake_gain_v2", "grupo_segmented_v3", order=['Improve', 'Not Improve', 'Worsen'])

█ mejora_frq_quality_ceiling_effect.pdf   ·  grouping: grupo_segmented_v3   ·  metric: mejora_hake_gain_v2 (proportions)
   Q: does the improvement-category mix differ between groups?
   Chi-square   χ² = 7.803, df = 10, n = 145   p = 0.6481 n.s.   Cramér's V = 0.164 (small)
   %Improve by group: ExpAA=86%, ExpAB=71%, ExpBA=91%, ExpBB=80%, ExpNotUsed=75%, control=83%
   most improving: ExpBA (91%)   |   least: ExpAB (71%)
   note: 56% of cells expected <5 (χ² approximate)
   → No significant difference in the improvement mix.



### Figure 14

**`mejora_nota_pre.pdf`** — by interaction quality, within each baseline stratum.

In [17]:
test_mejora_across_baseline_strata(forms_interactions_df)

TEST 3  -  is the improvement mix (Improve / Not Improve / Worsen) the SAME
           across the three baseline panels (score_tc_cat_pre = Low / Medium / High)
           of mejora_nota_pre.pdf, collapsing groups within each panel?
   metric : proportion in each mejora category
   test   : Chi-square of independence on the 3 (baseline) x 3 (mejora) table
            H0: the mejora mix is the same across the three baseline strata

        stratum        Improve   Not Improve        Worsen     n
            Low          92.1%          3.2%          4.8%      63
         Medium          72.0%         14.6%         13.4%      82
           High          52.6%         18.4%         28.9%      38

     χ² = 21.340, df = 4, n = 183   p = 0.0003 ***   Cramer's V = 0.241 (small)
     note: 11% of cells expected <5 (chi2 approximation)
     -> mejora mix DIFFERS across baseline strata


{'chi2': np.float64(21.34016405579001),
 'dof': 4,
 'p': np.float64(0.00027109166347584484),
 'cramers_v': np.float64(0.24146730455451967),
 'table': [[58.0, 2.0, 3.0], [59.0, 12.0, 11.0], [20.0, 7.0, 11.0]],
 'strata': ['Low', 'Medium', 'High'],
 'outcomes': ['Improve', 'Not Improve', 'Worsen']}

---
## 6 · Self-efficacy improvement (`self_confidence_comparison.pdf`)

Compare the **self-efficacy gain** (ta post − pre) between experimental and control.

### Figure 15 (a)

In [18]:
diff_2("self_confidence_comparison.pdf", "grupo", "self-efficacy gain (post−pre)",
       forms_interactions_df, "grupo", "experimental", "control",
       pre="score_ta_pre", post="score_ta_post")

█ self_confidence_comparison.pdf   ·  grouping: grupo   ·  metric: self-efficacy gain (post−pre)
   Q: is the improvement different between experimental and control?
   Mann-Whitney U   n=183   median experimental=+0.050 vs control=+0.080  (Δ=-0.030)
   U = 3444.0   p = 0.0399 *   rank-biserial r = -0.176 (small)
   → control is higher on self-efficacy gain (post−pre) — difference is small.



### Figure 15 (b)

In [19]:
diff_k("self_confidence_comparison_v4.pdf", "v4", "self-efficacy gain (post−pre)",
       forms_interactions_df, "grupo_segmented_v4",
       pre="score_ta_pre", post="score_ta_post")

█ self_confidence_comparison_v4.pdf   ·  grouping: grupo_segmented_v4   ·  metric: self-efficacy gain (post−pre)
   Q: does self-efficacy gain (post−pre) differ across the 4 groups?
   Kruskal-Wallis   H = 5.448, df = 3, n = 183   p = 0.1418 n.s.   ε² = 0.014 (small)
   group medians: control=+0.080, ExpNotUsed=+0.050, ExpB=+0.030, ExpA=+0.070
   highest: control (+0.080)   |   lowest: ExpB (+0.030)
   BIGGEST pairwise difference: control vs ExpB   U = 2142.0, r = +0.253 (small), p-Holm = 0.1563 n.s.
   (no pair survives Holm correction)



In [20]:
test_expa_ta_post(forms_interactions_df, target="ExpA")

TEST 2  -  does 'ExpA' hold HIGHER post self-efficacy (score_ta_post) than the other grupo_segmented_v4 groups?
   metric : continuous post self-efficacy score
   test   : Mann-Whitney U on score_ta_post, target vs each other group,
            Holm-adjusted    (positive r => target scores HIGHER)

     contrast                          n_t  n_c   med(t)  med(c)        U        r         p    p-Holm   effect
     ExpA vs ExpB                   25   36     0.58    0.54    501.5   +0.114    0.4529    0.4529 n.s.  small higher
     ExpA vs ExpNotUsed             25   27     0.58    0.50    430.5   +0.276    0.0892    0.1784 n.s.  small higher
     ExpA vs control                25   95     0.58    0.48   1521.5   +0.281    0.0308    0.0924 n.s.  small higher

     -> 'ExpA' is significantly HIGHER on self-efficacy (post) than 0/3 of the other groups (Holm-adjusted).


{'target': 'ExpA',
 'contrasts': [{'comparator': 'ExpB',
   'n_t': 25,
   'n_c': 36,
   'med_t': 0.58,
   'med_c': 0.535,
   'U': 501.5,
   'r': np.float64(0.11444444444444435),
   'p': np.float64(0.4529148899950348),
   'p_holm': np.float64(0.4529148899950348)},
  {'comparator': 'ExpNotUsed',
   'n_t': 25,
   'n_c': 27,
   'med_t': 0.58,
   'med_c': 0.5,
   'U': 430.5,
   'r': np.float64(0.27555555555555555),
   'p': np.float64(0.08920616327301238),
   'p_holm': np.float64(0.17841232654602476)},
  {'comparator': 'control',
   'n_t': 25,
   'n_c': 95,
   'med_t': 0.58,
   'med_c': 0.48,
   'U': 1521.5,
   'r': np.float64(0.2812631578947369),
   'p': np.float64(0.030802217861101443),
   'p_holm': np.float64(0.09240665358330433)}],
 'wins': 0}

---
## 7 · Cognitive load — does post load differ between groups?

Load is measured only post, so we compare the **post score** between groups (not an
improvement). Three subscales: relevant/germane, extraneous, intrinsic.

### Figure 17 (a)

**`tcc_comparison.pdf`** — each subscale, experimental vs control.

In [21]:
for c, lab in [("score_tcc_rel_post", "relevant load"),
                  ("score_tcc_ext_post", "extraneous load"),
                  ("score_tcc_int_post", "intrinsic load")]:
    diff_2(f"tcc_comparison.pdf [{lab}]", "grupo", lab, forms_interactions_df,
           "grupo", "experimental", "control", metric=c)

█ tcc_comparison.pdf [relevant load]   ·  grouping: grupo   ·  metric: relevant load
   Q: is the improvement different between experimental and control?
   Mann-Whitney U   n=183   median experimental=+0.730 vs control=+0.730  (Δ=+0.000)
   U = 4490.0   p = 0.3863 n.s.   rank-biserial r = +0.074 (negligible)
   → no significant between-group difference (negligible effect).

█ tcc_comparison.pdf [extraneous load]   ·  grouping: grupo   ·  metric: extraneous load
   Q: is the improvement different between experimental and control?
   Mann-Whitney U   n=183   median experimental=+0.430 vs control=+0.300  (Δ=+0.130)
   U = 5534.0   p = 0.0002 ***   rank-biserial r = +0.324 (medium)
   → experimental is higher on extraneous load — difference is medium.

█ tcc_comparison.pdf [intrinsic load]   ·  grouping: grupo   ·  metric: intrinsic load
   Q: is the improvement different between experimental and control?
   Mann-Whitney U   n=183   median experimental=+0.415 vs control=+0.300  (Δ=+0.115)

### Figure 17 (b)

**`tcc_comparison_quality.pdf`** — each subscale across interaction quality (`grupo_segmented_v4`).

In [22]:
for c, lab in [("score_tcc_rel_post", "relevant load"),
                  ("score_tcc_ext_post", "extraneous load"),
                  ("score_tcc_int_post", "intrinsic load")]:
    diff_k(f"tcc_comparison_quality.pdf [{lab}]", "v4", lab, forms_interactions_df,
           "grupo_segmented_v4", metric=c)

█ tcc_comparison_quality.pdf [relevant load]   ·  grouping: grupo_segmented_v4   ·  metric: relevant load
   Q: does relevant load differ across the 4 groups?
   Kruskal-Wallis   H = 5.731, df = 3, n = 183   p = 0.1255 n.s.   ε² = 0.015 (small)
   group medians: ExpNotUsed=+0.730, control=+0.730, ExpA=+0.830, ExpB=+0.700
   highest: ExpA (+0.830)   |   lowest: ExpB (+0.700)
   BIGGEST pairwise difference: ExpA vs ExpB   U = 586.0, r = +0.302 (medium), p-Holm = 0.2307 n.s.
   (no pair survives Holm correction)

█ tcc_comparison_quality.pdf [extraneous load]   ·  grouping: grupo_segmented_v4   ·  metric: extraneous load
   Q: does extraneous load differ across the 4 groups?
   Kruskal-Wallis   H = 14.833, df = 3, n = 183   p = 0.0020 **   ε² = 0.066 (medium)
   group medians: ExpNotUsed=+0.430, ExpA=+0.430, ExpB=+0.430, control=+0.300
   highest: ExpNotUsed (+0.430)   |   lowest: control (+0.300)
   BIGGEST pairwise difference: ExpNotUsed vs control   U = 1744.5, r = +0.360 (medium), p-H

### Figure 18

**`tcc_mejora.pdf`** — each subscale across improvement outcome (`mejora_hake_gain_v2`).

In [23]:
for c, lab in [("score_tcc_rel_post", "relevant load"),
                  ("score_tcc_ext_post", "extraneous load"),
                  ("score_tcc_int_post", "intrinsic load")]:
    diff_k(f"tcc_mejora.pdf [{lab}]", "mejora", lab, forms_interactions_df,
           "mejora_hake_gain_v2", metric=c)

█ tcc_mejora.pdf [relevant load]   ·  grouping: mejora_hake_gain_v2   ·  metric: relevant load
   Q: does relevant load differ across the 3 groups?
   Kruskal-Wallis   H = 11.160, df = 2, n = 183   p = 0.0038 **   ε² = 0.051 (small)
   group medians: Worsen=+0.670, Not Improve=+0.670, Improve=+0.770
   highest: Improve (+0.770)   |   lowest: Worsen (+0.670)
   BIGGEST pairwise difference: Worsen vs Improve   U = 1106.5, r = -0.354 (medium), p-Holm = 0.0147 *
   significant pairs (Holm): Improve>Worsen r=+0.35

█ tcc_mejora.pdf [extraneous load]   ·  grouping: mejora_hake_gain_v2   ·  metric: extraneous load
   Q: does extraneous load differ across the 3 groups?
   Kruskal-Wallis   H = 19.389, df = 2, n = 183   p = 0.0001 ***   ε² = 0.097 (medium)
   group medians: Improve=+0.300, Worsen=+0.500, Not Improve=+0.430
   highest: Worsen (+0.500)   |   lowest: Improve (+0.300)
   BIGGEST pairwise difference: Improve vs Worsen   U = 811.0, r = -0.526 (large), p-Holm = 0.0001 ***
   significan